# Davinci — DPO Preference Optimization

This notebook fine-tunes a verified SFT checkpoint using human-reviewed `prompt` / `chosen` / `rejected` comparisons. It uses a frozen reference model and the canonical DPO objective from `src/davinci/preference.py`.

**Requirements:** a completed SFT checkpoint, the exact tokenizer used by that checkpoint, a preference JSONL file, and a manifest with explicit license/provenance review and an artifact SHA-256. Do not fabricate preference labels or infer that a response is correct merely because it is preferred.

Start with a small checkpoint and a short pilot. Full 500M DPO may require substantial accelerator memory because this baseline keeps both policy and reference weights resident.

## 1. Install lightweight dependencies

In [ ]:
!pip -q install sentencepiece tqdm

In [ ]:
import json, hashlib, random, subprocess, sys, time
from pathlib import Path

import torch
from torch.utils.data import DataLoader
import sentencepiece as spm

SEED=42
random.seed(SEED)
torch.manual_seed(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('device:',DEVICE)
if DEVICE=='cuda':
    print(torch.cuda.get_device_name(0), 'VRAM GiB', round(torch.cuda.get_device_properties(0).total_memory/2**30,2))

## 2. Clone the canonical repository and import tested helpers

In [ ]:
REPO_DIR=Path('/content/Davinci')
if not (REPO_DIR/'.git').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/keyhmoham/Davinci.git',str(REPO_DIR)],check=True)
else:
    subprocess.run(['git','-C',str(REPO_DIR),'pull','--ff-only'],check=True)
sys.path.insert(0,str(REPO_DIR))
REPO_COMMIT=subprocess.check_output(['git','-C',str(REPO_DIR),'rev-parse','HEAD'],text=True).strip()
from src.davinci import DavinciForCausalLM
from src.davinci.data_pipeline import load_dataset_manifest, validate_dataset_manifest, verify_dataset_artifact, iter_jsonl
from src.davinci.preference import validate_preference_pair, preference_split, sequence_log_probs, dpo_loss
print('repository commit:',REPO_COMMIT)

## 3. Mount Drive and configure immutable inputs

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PREFERENCE_JSONL=Path('/content/drive/MyDrive/Davinci/preference_data/processed/preferences.jsonl')
PREFERENCE_MANIFEST=Path('/content/drive/MyDrive/Davinci/preference_data/processed/preferences_manifest.json')
TOKENIZER_MODEL=Path('/content/drive/MyDrive/Davinci/tokenizer/davinci_sentencepiece.model')
BASE_SFT_DIR=Path('/content/drive/MyDrive/Davinci/sft_runs/run_001/model')
OUTPUT_DIR=Path('/content/drive/MyDrive/Davinci/dpo_runs/run_001')
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
for path in (PREFERENCE_JSONL,PREFERENCE_MANIFEST,TOKENIZER_MODEL,BASE_SFT_DIR/'config.json',BASE_SFT_DIR/'model.pt'):
    if not path.is_file(): raise FileNotFoundError(f'Required DPO input is missing: {path}')
manifest=load_dataset_manifest(PREFERENCE_MANIFEST)
validate_dataset_manifest(manifest,intended_purpose='preference')
verify_dataset_artifact(PREFERENCE_JSONL,manifest,hash_field='preference_jsonl_sha256')
print('Preference manifest verified:',manifest['dataset_id'],manifest['dataset_revision'])

## 4. Validate preference pairs and split by prompt

All comparisons sharing a normalized prompt are assigned to the same partition. The split is deterministic; review the source data and manually inspect preference-label quality before training.

In [ ]:
VALIDATION_FRACTION=0.02
pairs=[]
seen=set()
rejected=[]
for line_number,record in enumerate(iter_jsonl(PREFERENCE_JSONL),1):
    try:
        pair=validate_preference_pair(record)
        key=hashlib.sha256((pair['prompt']+'\\0'+pair['chosen']+'\\0'+pair['rejected']).encode('utf-8')).hexdigest()
        if key in seen: continue
        seen.add(key)
        pair['split']=preference_split(pair['prompt'],validation_fraction=VALIDATION_FRACTION,seed=SEED)
        pairs.append(pair)
    except Exception as exc:
        rejected.append({'line':line_number,'reason':str(exc)})
train_pairs=[p for p in pairs if p['split']=='train']
val_pairs=[p for p in pairs if p['split']=='validation']
if not train_pairs or not val_pairs:
    raise ValueError('Preference split is empty; add more reviewed comparisons or adjust VALIDATION_FRACTION.')
(OUTPUT_DIR/'rejected_rows.json').write_text(json.dumps(rejected,ensure_ascii=False,indent=2),encoding='utf-8')
print({'valid_pairs':len(pairs),'train':len(train_pairs),'validation':len(val_pairs),'rejected':len(rejected)})

## 5. Load the frozen tokenizer and build response-only examples

In [ ]:
tokenizer=spm.SentencePieceProcessor(model_file=str(TOKENIZER_MODEL))
if tokenizer.eos_id()<0 or tokenizer.pad_id()<0:
    raise ValueError('Tokenizer must define EOS and PAD IDs.')

MAX_LENGTH=1024

def encode_pair(pair):
    prefix='<|user|>\\n'+pair['prompt']+'\\n<|assistant|>\\n'
    prompt_ids=tokenizer.encode(prefix,out_type=int)
    chosen_ids=tokenizer.encode(pair['chosen'],out_type=int)+[tokenizer.eos_id()]
    rejected_ids=tokenizer.encode(pair['rejected'],out_type=int)+[tokenizer.eos_id()]
    if not prompt_ids or not chosen_ids or not rejected_ids:
        raise ValueError('Empty tokenized prompt or response')
    def pack(response_ids):
        ids=prompt_ids+response_ids
        if len(ids)>MAX_LENGTH:
            ids=ids[-MAX_LENGTH:]
            # Keep at least one supervised response token after truncation.
            target_start=max(0,len(ids)-len(response_ids))
        else:
            target_start=len(prompt_ids)
        labels=[-100]*target_start+ids[target_start:]
        return {'input_ids':ids,'labels':labels}
    return {'chosen':pack(chosen_ids),'rejected':pack(rejected_ids)}

encoded_train=[encode_pair(p) for p in train_pairs]
encoded_val=[encode_pair(p) for p in val_pairs]
PAD_ID=tokenizer.pad_id()

def collate_side(items,side):
    rows=[item[side] for item in items]
    width=max(len(row['input_ids']) for row in rows)
    ids=[]; labels=[]; masks=[]
    for row in rows:
        n=width-len(row['input_ids'])
        ids.append(row['input_ids']+[PAD_ID]*n)
        labels.append(row['labels']+[-100]*n)
        masks.append([1]*len(row['input_ids'])+[0]*n)
    return (torch.tensor(ids,dtype=torch.long),torch.tensor(labels,dtype=torch.long),torch.tensor(masks,dtype=torch.bool))

def collate_pairs(items):
    return {'chosen':collate_side(items,'chosen'),'rejected':collate_side(items,'rejected')}

## 6. Load policy and frozen reference models

Both models must use the same base SFT checkpoint and tokenizer. The reference is frozen and evaluated without gradients. This full-model baseline is memory-intensive; do not change checkpoint provenance silently.

In [ ]:
policy=DavinciForCausalLM.from_pretrained(BASE_SFT_DIR,map_location='cpu').to(DEVICE)
reference=DavinciForCausalLM.from_pretrained(BASE_SFT_DIR,map_location='cpu').to(DEVICE)
reference.eval()
for parameter in reference.parameters(): parameter.requires_grad_(False)
if policy.config.vocab_size!=tokenizer.get_piece_size() or reference.config.vocab_size!=tokenizer.get_piece_size():
    raise ValueError('Model and tokenizer vocabulary sizes differ.')
optimizer=torch.optim.AdamW(policy.parameters(),lr=5e-7,weight_decay=0.0)
BATCH_SIZE=1
GRAD_ACCUM_STEPS=4
MAX_STEPS=100
BETA=0.1
LOG_EVERY=10
train_loader=DataLoader(encoded_train,batch_size=BATCH_SIZE,shuffle=True,collate_fn=collate_pairs)
val_loader=DataLoader(encoded_val,batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate_pairs)
print('policy parameters:',sum(p.numel() for p in policy.parameters()))

## 7. Compute DPO loss from policy/reference sequence log-probabilities

In [ ]:
def score_side(model,batch,grad):
    ids,labels,mask=(x.to(DEVICE) for x in batch)
    if grad:
        output=model(ids)
    else:
        with torch.no_grad(): output=model(ids)
    return sequence_log_probs(output['logits'],labels,attention_mask=mask)[0]

def dpo_batch(batch,train):
    policy_chosen=score_side(policy,batch['chosen'],grad=train)
    policy_rejected=score_side(policy,batch['rejected'],grad=train)
    reference_chosen=score_side(reference,batch['chosen'],grad=False)
    reference_rejected=score_side(reference,batch['rejected'],grad=False)
    return dpo_loss(policy_chosen,policy_rejected,reference_chosen,reference_rejected,beta=BETA)

@torch.no_grad()
def evaluate_dpo(loader,max_batches=50):
    was_training=policy.training
    policy.eval(); values=[]; accuracies=[]
    try:
        for index,batch in enumerate(loader):
            if index>=max_batches: break
            result=dpo_batch(batch,train=False)
            values.append(float(result['loss'].cpu()))
            accuracies.append(float(result['preference_accuracy'].cpu()))
    finally:
        policy.train(was_training)
    if not values: raise ValueError('No validation preference batches were evaluated.')
    return {'loss':sum(values)/len(values),'preference_accuracy':sum(accuracies)/len(accuracies)}

history=[]
policy.train(); optimizer.zero_grad(set_to_none=True)
step=0; micro=0
while step<MAX_STEPS:
    progressed=False
    for batch in train_loader:
        progressed=True
        result=dpo_batch(batch,train=True)
        (result['loss']/GRAD_ACCUM_STEPS).backward()
        micro+=1
        if micro%GRAD_ACCUM_STEPS==0:
            torch.nn.utils.clip_grad_norm_(policy.parameters(),1.0)
            optimizer.step(); optimizer.zero_grad(set_to_none=True)
            step+=1
            if step%LOG_EVERY==0:
                metrics=evaluate_dpo(val_loader)
                row={'step':step,'train_loss':float(result['loss'].detach().cpu()),**metrics}
                history.append(row); print(row)
            if step>=MAX_STEPS: break
    if not progressed: raise RuntimeError('Training preference loader is empty.')
    if micro%GRAD_ACCUM_STEPS:
        # Drop the incomplete accumulation window rather than applying a differently scaled update.
        optimizer.zero_grad(set_to_none=True)
        micro-=micro%GRAD_ACCUM_STEPS
    if step==0 and micro==0: raise RuntimeError('No complete gradient-accumulation window was produced.')
print('DPO pilot complete; steps:',step)

## 8. Save checkpoint and run manifest

In [ ]:
def file_sha256(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda:handle.read(1024*1024),b''):
            digest.update(chunk)
    return digest.hexdigest()

policy.save_pretrained(OUTPUT_DIR/'model')
base_hash=file_sha256(BASE_SFT_DIR/'model.pt')
tokenizer_hash=file_sha256(TOKENIZER_MODEL)
data_hash=file_sha256(PREFERENCE_JSONL)
manifest_hash=file_sha256(PREFERENCE_MANIFEST)
checkpoint={
    'model':policy.state_dict(),'optimizer':optimizer.state_dict(),'step':step,
    'repo_commit':REPO_COMMIT,'base_sft_model_sha256':base_hash,
    'tokenizer_sha256':tokenizer_hash,'preference_jsonl_sha256':data_hash,
    'preference_manifest_sha256':manifest_hash,'beta':BETA,
    'history':history,'model_config':policy.config.__dict__,
}
torch.save(checkpoint,OUTPUT_DIR/'dpo_checkpoint.pt')
run_manifest={
    'repo_commit':REPO_COMMIT,'base_sft_model_sha256':base_hash,
    'tokenizer_sha256':tokenizer_hash,'preference_jsonl_sha256':data_hash,
    'preference_manifest_sha256':manifest_hash,'train_pairs':len(train_pairs),
    'validation_pairs':len(val_pairs),'steps':step,'beta':BETA,
    'batch_size':BATCH_SIZE,'gradient_accumulation_steps':GRAD_ACCUM_STEPS,
    'device':DEVICE,'history':history,
    'limitations':['full-model DPO baseline','no LoRA/QLoRA','preference labels require human quality review','pilot metrics are not benchmark claims'],
}
(OUTPUT_DIR/'run_manifest.json').write_text(json.dumps(run_manifest,ensure_ascii=False,indent=2),encoding='utf-8')
print('Saved DPO artifacts:',OUTPUT_DIR)

## 9. Evaluation and interpretation

Compare the DPO checkpoint with the SFT baseline on held-out prompts, Persian and English separately. Track win rate, response quality, refusal behavior, tool-call validity, and regression on general language tasks. Do not use the same prompts for preference training and evaluation.

This notebook is a full-model DPO pilot scaffold. It does not implement LoRA/QLoRA, distributed training, or automatic preference-label verification. Run only after a real, reviewed preference dataset and SFT checkpoint exist.